# DeFiPy: Python SDK for DeFi Analytics
## Chapter 6: Yield and Risk Assessment

### Listing 6.1: Calculating APY for a 0.2% daily return

In [6]:
from defipy import CompoundReturn, ConstantIPS

# Step 1: Calculate IPS
returns_calc = 0.002  # 0.2% return over 1 day
total_seconds = 86400  # 1 day in seconds
ips = ConstantIPS().calc_ips(1, 1 + returns_calc, total_seconds)

# Step 2: Compute APY
final_amount = CompoundReturn().apply(1, ConstantIPS.SECONDS_YEAR, ips)
APY = final_amount - 1
print(f"APY: {APY * 100:.2f}%")

APY: 107.46%


### Listing 6.2: Uniswap V2 setup for impermanent loss

In [7]:
from defipy import *

# Step 1: Simulate price
n_steps = 500
start_price = 10
bm = BrownianModel(start_price)
p_arr = bm.gen_gbms(0.1, 0.5, n_steps-1, 1).flatten()

# Step 2: Setup Uniswap V2 pool
user_nm = 'user'
tkn1_amount = 1000
tkn2_amount = p_arr[0]*tkn1_amount

tkn1 = ERC20('TKN1', "0x111")
tkn2 = ERC20('TKN2', "0x09")
exchg_data = UniswapExchangeData(tkn0 = tkn1, tkn1 = tkn2, symbol="LP", address="0x011")

factory = UniswapFactory("pool factory", "0x2")
lp = factory.deploy(exchg_data)
Join().apply(lp, user_nm, tkn1_amount, tkn2_amount)
lp.summary()

# Step 3: Add liquidity and initialize imp. loss calculator
out = AddLiquidity().apply(lp, tkn1, user_nm, 10)
dL = lp.get_last_liquidity_deposit()
iLoss = UniswapImpLoss(lp, dL)
lp.summary()

Exchange TKN1-TKN2 (LP)
Reserves: TKN1 = 1000.0, TKN2 = 10000.0
Liquidity: 3162.2776601683795 

Exchange TKN1-TKN2 (LP)
Reserves: TKN1 = 1010.0, TKN2 = 10100.0
Liquidity: 3193.900436770063 



### Listing 6.3: Uniswap V2 impermanent loss simulation

In [8]:
# Step 4: Run simulation
arb = CorrectReserves(lp, x0 = p_arr[0])
TKN_amt = TokenDeltaModel(100)
pTKN1_TKN2_arr = []; 

for k in range(n_steps):

    # *****************************
    # ***** Random Swapping ******
    # *****************************
    Swap().apply(lp, tkn1, user_nm, TKN_amt.delta())
    Swap().apply(lp, tkn2, user_nm, p_arr[k]*TKN_amt.delta())
    
    # *****************************
    # ***** Rebalance ******
    # *****************************
    arb.apply(p_arr[k])

    # *****************************
    # ******* Data Capture ********
    # *****************************
    pTKN1_TKN2_arr.append(LPQuote().get_price(lp, tkn1))

lp.summary()

# Step 5: Retrieve imp. loss calculation
iloss_calc = iLoss.apply(fees = False)
returns_calc = iLoss.apply(fees = True)
print(f'Imp Loss: {100*iloss_calc:.2f}%, returns: {100*returns_calc:.2f} %')

Exchange TKN1-TKN2 (LP)
Reserves: TKN1 = 1398.0498037201144, TKN2 = 10310.204792929326
Liquidity: 3665.263284863321 

Imp Loss: -1.15%, returns: 1.80 %


### Listing 6.4: Uniswap V3 setup for impermanent loss

In [9]:
# Step 2: Setup Uniswap V3 pool
tkn1 = ERC20('TKN1', "0x09")
tkn2 = ERC20('TKN2', "0x111")

exchg_data = UniswapExchangeData(tkn0 = tkn1, tkn1 = tkn2, symbol="LP",
                                   address="0x011", version = 'V3',
                                   tick_spacing = 60,
                                   fee = 3000)

factory = UniswapFactory("ETH pool factory", "0x2")
lp = factory.deploy(exchg_data)

Join().apply(lp, user_nm, tkn1_amount, tkn2_amount, -887220, 887220)
lp.summary()

# Step 3: Add liquidity and initialize imp. loss calculator
tick_size = 10000
lwr_tick = UniV3Helper().get_price_tick(lp, -1, lp.get_price(tkn1), tick_size)
upr_tick = UniV3Helper().get_price_tick(lp, 1, lp.get_price(tkn1), tick_size)  
out = AddLiquidity().apply(lp, tkn1, user_nm, 10, lwr_tick, upr_tick)
lp.summary()

dL = lp.last_liquidity_deposit
iLoss = UniswapImpLoss(lp, dL, lwr_tick, upr_tick)

Exchange TKN1-TKN2 (LP)
Real Reserves:   TKN1 = 999.9999999999999, TKN2 = 10000.0
Gross Liquidity: 3162.277660168379 

Exchange TKN1-TKN2 (LP)
Real Reserves:   TKN1 = 1009.9999999999999, TKN2 = 10100.262769858964
Gross Liquidity: 3242.81759104987 



### Listing 6.5: Uniswap V3 impermanent loss simulation

In [10]:
tick_size = 20000
arb = CorrectReserves(lp, x0 = p_arr[0])
TKN_amt = TokenDeltaModel(100)
pTKN1_TKN2_arr = []; 

for k in range(n_steps):

    # *****************************
    # ***** Random Swapping ******
    # *****************************
    Swap().apply(lp, tkn1, user_nm, TKN_amt.delta())
    Swap().apply(lp, tkn2, user_nm, p_arr[k]*TKN_amt.delta())
    
    # *****************************
    # ***** Rebalance ******
    # *****************************

    lwr_tick = UniV3Helper().get_price_tick(lp, -1, lp.get_price(tkn1), tick_size)
    upr_tick = UniV3Helper().get_price_tick(lp, 1, lp.get_price(tkn1), tick_size)   
    arb.apply(p_arr[k], lwr_tick, upr_tick)

    # *****************************
    # ******* Data Capture ********
    # *****************************
    pTKN1_TKN2_arr.append(LPQuote().get_price(lp, tkn1))

lp.summary()

# Step 5: Retrieve imp. loss calculation
lwr_tick = UniV3Helper().get_price_tick(lp, -1, lp.get_price(tkn1), tick_size)
upr_tick = UniV3Helper().get_price_tick(lp, 1, lp.get_price(tkn1), tick_size)   
iloss_calc = iLoss.apply( lwr_tick, upr_tick, fees = False)
returns_calc = iLoss.apply( lwr_tick, upr_tick, fees = True)
print(f'Imp Loss: {100*iloss_calc:.2f}%, returns: {100*returns_calc:.2f} %')

Exchange TKN1-TKN2 (LP)
Real Reserves:   TKN1 = 1384.4960527071728, TKN2 = 10164.004703424582
Gross Liquidity: 3931.932689542414 

Imp Loss: -2.41%, returns: 57.54 %
